<a href="https://colab.research.google.com/github/Fiona-ali/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task
**Lane:** Refresh / Content Opportunity Scoring | Week 2

## 1. My lane as an ML task (type)
**Task: ranking/scoring.** The decision is which pages a content editor should review first for possible refresh. The output is a ranked review queue, not automatic content edits. A classification proxy (observed decline yes/no) helps evaluate ranking. False positives waste editor hours; false negatives leave declining pages unreviewed.

In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/Fiona-ali/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
print("Rows, columns:", df.shape)
print("Clients:", df.client_id.nunique())

Rows, columns: (30000, 44)
Clients: 32


## 2. Target or proxy
**Observed-decline teaching proxy:** 1 if `trend_direction == "down"`, else 0. In the starter dataset, `down` denotes an impressions drop greater than 20% between adjacent 30-day windows. This is a rule-defined label based on an observed comparison, **not** an independently observed future outcome or evidence that refreshing a page causes recovery. `trend_direction`, `trend_pct`, and `impressions_last_30d` are excluded from candidate predictors to avoid direct leakage. A future prediction task needs later-window outcomes from the warehouse.

In [2]:
df["observed_decline_proxy"] = (df["trend_direction"] == "down").astype(int)
print("Proxy counts:")
print(df["observed_decline_proxy"].value_counts().sort_index())
print("Share declining:", round(df["observed_decline_proxy"].mean(), 3))

Proxy counts:
observed_decline_proxy
0    13738
1    16262
Name: count, dtype: int64
Share declining: 0.542


## 3. Success metric
**Precision@50** measures the proportion of the 50 highest-ranked pages that show the observed decline proxy. For example, 35/50 = 0.70. A useful future model must outperform a transparent fixed rule on held-out clients, not only on training data. A 100-impression previous-window floor focuses attention on measurable pages. This snapshot metric does **not** measure future performance or refresh impact.

In [3]:
eligible = df.loc[df["impressions_prev_30d"].fillna(0) >= 100].copy()
eligible["fixed_rule_score"] = eligible["days_since_last_update"].fillna(0)
ranked = eligible.sort_values("fixed_rule_score", ascending=False)
print("Eligible pages:", len(eligible))
if len(ranked) >= 50:
    print("Illustrative fixed-rule Precision@50:", round(ranked.head(50)["observed_decline_proxy"].mean(), 3))
else:
    print("Not enough pages for Precision@50")

Eligible pages: 18010
Illustrative fixed-rule Precision@50: 0.86


## 4. The unit of analysis, as a real dataframe
**One row = one pseudonymized content page** in a 90-day snapshot, not one day or query. `content_id` and `client_id` are identifiers for joins, grouping and client-holdout evaluation, **never** predictive features. The following real dataframe is the lane slice (pages with at least 100 previous-window impressions). Missing search volume must be flagged rather than blindly treated as zero.

In [4]:
cols = ["content_id", "client_id", "content_type", "content_age_days", "days_since_last_update", "impressions_prev_30d", "search_volume", "observed_decline_proxy"]
review_slice = eligible[cols].copy()
print("Slice shape:", review_slice.shape)
print("Duplicate content IDs:", review_slice.content_id.duplicated().sum())
assert review_slice.content_id.is_unique
display(review_slice.head(10))

Slice shape: (18010, 8)
Duplicate content IDs: 0


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_prev_30d,search_volume,observed_decline_proxy
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,987,10.0,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,5915,90.0,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,6089,0.0,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,4206,10.0,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,6452,0.0,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,1009,720.0,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,632,590.0,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,13828,0.0,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,356,0.0,1
10,content_d8ee6cc6d642,client_19581e27de,keyword article,329,104,6441,20.0,0


## 5. Why ML beats a fixed rule here
A rule such as “review the oldest pages” ignores search demand, content type, age, and traffic history. These factors may interact: an old low-demand page might be less urgent than a newer page with significant search activity. ML is **worth testing**, not automatically superior: compare its ranked queue with the fixed-rule Precision@50 using held-out clients. If it does not consistently improve decisions, keep the simpler rule. Do not use the label-generating trend fields as model inputs.

**ML loop:** Frame editor decision → collect anonymized page-level signals → define observed-decline proxy → rank with a fixed-rule baseline → later train and validate on held-out clients → give editor a review queue → collect feedback.

**One-paragraph frame:** For content editors deciding which pages to investigate first, I propose a page-level ranking score using search and content metadata, with observed impression decline as an initial teaching proxy and Precision@50 as the metric. False alarms waste editor time, while missed declines delay investigation. Multiple interacting signals make ML worth evaluating, but results are only directional decision support, not proof of Google ranking mechanism or refresh effectiveness.

In [5]:
proposed_features = ["impressions_prev_30d", "content_age_days", "days_since_last_update", "content_type", "search_volume"]
for leaked in ["trend_direction", "trend_pct", "impressions_last_30d", "observed_decline_proxy"]:
    assert leaked not in proposed_features
print("Proposed non-label inputs:", proposed_features)
display(ranked[["content_id", "days_since_last_update", "impressions_prev_30d", "observed_decline_proxy"]].head(10))

Proposed non-label inputs: ['impressions_prev_30d', 'content_age_days', 'days_since_last_update', 'content_type', 'search_volume']


,content_id,days_since_last_update,impressions_prev_30d,observed_decline_proxy
21984,content_02b0d6e30129,313,159,1
7452,content_72496874f806,301,125,1
26840,content_7f116ae1f6f5,301,339,1
1227,content_4f241bad48a3,236,187,1
6233,content_ba00ffc6318c,211,113,1
26799,content_77d4d5930e5e,194,527,1
5327,content_fe16a55cd13d,194,1562,1
16751,content_cf56e2e2e282,194,26791,1
11489,content_5feee3994adb,194,2670,1
16514,content_7368877ea310,194,20472,1


## 6. Self-check
- [x] Named the ranking/scoring task, target/proxy, metric and editor action.
- [x] Explained error costs, ML-vs-rule hypothesis, and proxy limitations.
- [x] Added code to load the real starter CSV, show one row per content page, and calculate baseline Precision@50.
- [x] Used only pseudonymized identifiers and safe, directional claims.
- [ ] **Run all cells in Colab and inspect outputs.**
- [ ] **Save executed notebook in GitHub under `work/notebooks/` and submit repository URL.**

**Personal reflection:** I chose a ranking task because the editor needs a prioritized list, not just a yes/no label. The editor decides what to change after investigating the page.